In [1]:
import geemap
import utils as F
import ee
import os
import concurrent.futures
import glob
import rasterio
from rasterio.merge import merge
import geopandas as gpd
from shapely.geometry import mapping
from tqdm import tqdm
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

In [12]:
path = 'E:\Tide_flat_and_elevation_mapping/fishnet_with_intertidal\grid_with_mgrs.shp'
gdf = gpd.read_file(path)
mask = (gdf['province']==5) & (gdf['MGRS_Count']==1)
JS_grid = gdf[mask]


In [ ]:
ee.Initialize()



def shapely_to_ee_geometry(shapely_geom):
    geo_json = mapping(shapely_geom)  # shapely 转 GeoJSON 格式 dict
    return ee.Geometry(geo_json)

def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')
  
    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    
    return img.set('cloud_per', cloud_per)
  
def CLIP(img):
    return img.clip(roi)

def cut_tiles_and_download(roi, output_dir, feature_bands, scale, max_workers):
    coords = roi.bounds().coordinates().getInfo()[0]
    minLon, minLat = coords[0]
    maxLon, maxLat = coords[2]

    # 经纬度换算（粗略）
    tile_size_deg = 5000 / 111320

    tiles = []
    lat = minLat
    while lat < maxLat:
        lon = minLon
        while lon < maxLon:
            lon2 = min(lon + tile_size_deg, maxLon)
            lat2 = min(lat + tile_size_deg, maxLat)
            tiles.append(ee.Geometry.BBox(lon, lat, lon2, lat2))
            lon += tile_size_deg
        lat += tile_size_deg

    print(f"将切成 {len(tiles)} 个切片进行下载...")
    #output_dir = "G:\GEEdownload2"
    os.makedirs(output_dir, exist_ok=True)

    def export_tile(i_tile):
        i, tile = i_tile
        filename = os.path.join(output_dir, f"tile_{i+1}.tif")
       # print(f"开始导出 {filename} ...")
        geemap.ee_export_image(
            feature_bands,
            filename=filename,
            scale= scale,
            region=tile,
            file_per_band=False
        )
       # print(f"完成导出 {filename}")
        return filename

    # # 用线程池并发执行导出
    # with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    #     results = list(executor.map(export_tile, enumerate(tiles)))
        
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(
            executor.map(export_tile, enumerate(tiles)),
            total=len(tiles),
            desc="下载进度",
            unit="切片"
        ))

    print("✅ 所有切片导出完成！")

def Tile_mosaic(input_dir, output_dir ,No):
    
    src_files_to_mosaic = [rasterio.open(fp) for fp in glob.glob(f"{input_dir}/tile_*.tif")]
    mosaic, out_trans = merge(src_files_to_mosaic)

    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    merged_path = f"{output_dir}/JS_{No}_merged_image.tif"
    # os.makedirs(f"{output_dir}/JS_PJ", exist_ok=True)
    with rasterio.open(merged_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"✅ 已完成拼接，输出文件: {merged_path}")

# for i in range(1):
for i in range(18, len(JS_grid)):
    
    geom = JS_grid.geometry.iloc[i]
    roi = shapely_to_ee_geometry(geom)
    single_grid = JS_grid.iloc[[i]]
    MGRS_TILE = single_grid['MGRS_Name'].values[0]
    
    #--------------------base info----------------------
    start_date, end_date = '2024-01-01', '2025-01-01'
    cloud_limit = 40
    apply_MGRS_TILE = True
    output_dir = f"G:\GEEdownload2\JS\JS_{i}"
    PJ_output_dir = f"G:\GEEdownload2\JS\JS_PJ"
    
    
    
    #--------------------Feature calculate--------------
    S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)
    if apply_MGRS_TILE:
        print('--apply MGRS_TILE--')
        S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
        
    S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
            .map(CLIP).map(cal_cloud_per)\
            .filter(ee.Filter.gt('cloud_per', cloud_limit))
            
    S1 = ee.ImageCollection('COPERNICUS/S1_GRD').filter(ee.Filter.eq('platform_number', 'A')).filterBounds(roi)\
            .filterDate(start_date, end_date)\
            .map(CLIP).map(F.CR).mean()
    
    VV_mean = S1.select('VV')
    VH_mean = S1.select('VH')
    CR_mean = S1.select('CR')       
           
    L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
    L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)

    all_TF_collect = S2.merge(L8).merge(L9)
    all_TF_collect = ee.ImageCollection(all_TF_collect)
    sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
    sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
    Frequency = sum_TF.divide(sum_cloud).rename('Freq')
    # sum_TF = S2.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
    # sum_cloud = S2.select('cloud').toBands().reduce(ee.Reducer.sum())
    # Frequency = sum_TF.divide(sum_cloud).rename('Freq')

    S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
    Norm_S2_2022 = S2_2022.map(F.Norm)
    cloud_sum = S2_2022.select('cloud').toBands().reduce(ee.Reducer.sum())

    NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
    NDWI_mean = S2_2022.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDWI')
    MNDWI_mean = S2_2022.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_MNDWI')
    composit_b8A = S2_2022.select('B8A').reduce(ee.Reducer.sum()).divide(cloud_sum).divide(10000).rename('mean_B8A')
    NDVI_mean = S2_2022.select('NDVI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDVI')
    S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([1])).rename('S2_NDWI_min')

    S2_NDWI_norm_min = Norm_S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
    TF_mask = S2_NDWI_norm_min.lt(-0.1).And(Frequency.gt(0.05)).And(Frequency.lt(0.95)).clip(roi).unmask(0).rename('mask')

    feature_bands = Frequency.addBands(NDWI_mean).addBands(composit_b8A).addBands(MNDWI_mean)\
                    .addBands(NDVI_mean).addBands(NDWI_norm_min).addBands(VV_mean).addBands(VH_mean).addBands(CR_mean)\
                    .addBands(TF_mask)
    
    #----------------tile cut and download----------------
    cut_tiles_and_download(roi, output_dir, feature_bands, 10, 16)
    
    #---------------------- Mosaic -----------------------
    Tile_mosaic(output_dir ,PJ_output_dir, i)
    
    file_A = "G:\GEEdownload2\JS\JS_PJ\JS_0_merged_image.tif"
    other_files = ["G:\Tidal_range/tide_range_mean_2021.tif", "G:\Tidal_range/tide_max_2021.tif", "G:\Tidal_range/tide_min_2021.tif"]
    output_file = "G:\GEEdownload2\JS\JS_PJ\JS_0_merged_image.tif"

    # 打开A
    with rasterio.open(file_A) as src_A:
        A_data = src_A.read()              # shape = (bands, height, width)
        meta = src_A.meta.copy()
        A_crs = src_A.crs
        A_transform = src_A.transform
        height, width = src_A.height, src_A.width
        num_bands = src_A.count

    # 创建一个数组来存放最终数据
    final_data = np.empty((num_bands + len(other_files), height, width), dtype=A_data.dtype)
    final_data[:num_bands, :, :] = A_data

    # 遍历每个单波段 raster，将其重投影到 A 的大小和 CRS
    for idx, file in enumerate(other_files):
        with rasterio.open(file) as src:
            resampled = np.empty((height, width), dtype=src.dtypes[0])
            reproject(
                source=rasterio.band(src, 1),
                destination=resampled,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=A_transform,
                dst_crs=A_crs,
                resampling=Resampling.nearest
            )
            final_data[num_bands + idx, :, :] = resampled

    # 更新元信息，波段数增加
    meta.update(count=num_bands + len(other_files))

    # 写入新的多波段 raster
    with rasterio.open(output_file, 'w', **meta) as dst:
        for i in range(final_data.shape[0]):
            dst.write(final_data[i], i + 1)

    print(f"Done! 已生成 {output_file}")

Done! 已生成 G:\GEEdownload2\JS\JS_PJ\JS_0_merged_image.tif
--apply MGRS_TILE--
将切成 36 个切片进行下载...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...
Generating URL ...


下载进度:   0%|          | 0/36 [00:00<?, ?切片/s]

Please wait ...Downloading data from https://earthengine.googleapis.com/v1/projects/217688291046/thumbnails/9660009b40c6ef293ee8ddf6f412d5c8-f8e91291d138a1d5c9fcea04ee52e0d9:getPixels
Please wait ...
Please wait ...

Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_6.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_11.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_5.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_12.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_13.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_2.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_4.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_14.ti

下载进度:   3%|▎         | 1/36 [01:15<43:50, 75.17s/切片]

Data downloaded to G:\GEEdownload2\JS\JS_30\tile_1.tif
Generating URL ...
Data downloaded to G:\GEEdownload2\JS\JS_30\tile_16.tif
Generating URL ...
Please wait ...


下载进度:  22%|██▏       | 8/36 [01:17<03:20,  7.17s/切片]

... (14 outputs left out)

Data downloaded to G:\GEEdownload2\JS\JS_30\tile_34.tif


下载进度: 100%|██████████| 36/36 [03:11<00:00,  5.31s/切片]

Data downloaded to G:\GEEdownload2\JS\JS_30\tile_35.tif
✅ 所有切片导出完成！


✅ 已完成拼接，输出文件: G:\GEEdownload2\JS\JS_PJ/JS_30_merged_image.tif
